In [ ]:
import spacy, torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForMaskedLM
from tqdm import tqdm
from nltk.corpus import wordnet
from nltk.wsd import lesk

In [ ]:
original_dataset = load_dataset('glue', 'mrpc')
nlp = spacy.load("en_core_web_sm") # NER algorithm

model = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

model.eval()

#### Query Utility Functions
Takes a word and finds a suitable replacement

In [ ]:
def query_wordnet(word):
    syns = set()
    # pos=n retrieves only nouns (pos = Part-of-Speech)
    for synset in wordnet.synsets(word, pos='n'):
        for lemma in synset.lemmas():
            syns.add(lemma.name().replace('_',' ').lower())
    if word in syns: syns.remove(word)
    return syns

In [ ]:
def query_lesk(word, sen):
    syns = set()
    # pos=n retrieves only nouns (pos = Part-of-Speech)
    synset = lesk(sen.split(' '), word, pos='n')
    if not synset: return syns
    for lemma in synset.lemmas():
        syns.add(lemma.name().replace('_',' ').lower())
    if word in syns: syns.remove(word)
    return syns

In [ ]:
def query_bert(word, sen):
    inputs = tokenizer(sen, return_tensors='pt')
    # find the index for the mask token in the input token tensor
    mask_idx = (inputs.input_ids.squeeze() == tokenizer.mask_token_id).nonzero().item()
    with torch.no_grad():
        # model(**inputs).logits returns tensor 1 x sen_length x vocab_size
        # .squeeze() makes it sen_length x vocab_size
        # select row mask_idx for logits for mask across vocabulary
        mask_logits = model(**inputs).logits.squeeze()[mask_idx]
    # iterate mask logit tensor/list
    # index = word token, value = logit
    preds = {
        tokenizer.decode(idx): logit.item()
        for idx, logit in enumerate(mask_logits)
    }
    if word in preds: preds.pop(word)
    return preds

In [ ]:
' '.join(token.pos_ for token in nlp("Around 0335 GMT , Tab shares were up 19 cents , having earlier set a record high of $ 4.57 ."))

#### Prompt Utility Functions
Functions to construct and augment prompts

In [ ]:
def convert_to_prompt(s1, s2):
    return f"Sentence1: {s1}\nSentence2: {s2}\nDo these sentences mean the same thing? Respond with 1 if they do, or 0 if they don't."

In [ ]:
def augment_wordnet(sen):
    nouns = [token.text for token in nlp(sen) if token.pos_ == 'NOUN']
    if len(nouns) == 0: return sen

    for noun in nouns:
        syns = query_wordnet(noun)
        if len(syns) == 0: continue
        sen = sen.replace(noun, syns.pop(), 1)
    return sen

In [ ]:
def augment_lesk(sen):
    nouns = [token.text for token in nlp(sen) if token.pos_ == 'NOUN']
    if len(nouns) == 0: return sen

    for noun in nouns:
        syns = query_lesk(noun, sen)
        if len(syns) == 0: continue
        sen = sen.replace(noun, syns.pop(), 1)
    return sen

In [ ]:
def augment_bert(sen):
    nouns = [token.text for token in nlp(sen) if token.pos_ == 'NOUN']
    if len(nouns) == 0: return sen

    for noun in nouns:
        sen = sen.replace(noun, '[MASK]', 1)
        preds = query_bert(noun, sen)
        if len(preds) == 0:
            sen = sen.replace('[MASK]', noun, 1)
            continue
        newnoun = max(preds, key=lambda word: preds[word])
        sen = sen.replace('[MASK]', newnoun, 1)

    return sen

In [ ]:
def augment_hybrid(sen):
    nouns = [token.text for token in nlp(sen) if token.pos_ == 'NOUN']
    if len(nouns) == 0: return sen

    for noun in nouns:
        # run lesk
        syns = query_lesk(noun, sen)
        if len(syns) == 0: continue
        # run bert
        sen = sen.replace(noun, '[MASK]', 1)
        preds = query_bert(noun, sen)
        if len(preds) == 0:
            sen = sen.replace('[MASK]', noun, 1)
            continue
        # gets best word that is in syn
        best_syn = noun
        best_score = 0
        for syn in syns:
            if syn in preds and preds[syn] > best_score:
                best_syn = syn
                best_score = preds[syn]
        sen = sen.replace('[MASK]', best_syn, 1)
    return sen

#### Create New Dataset

In [ ]:
train_original = {'text': [], 'label': []}
train_augmented_wordnet = {'text': [], 'label': []}
train_augmented_lesk = {'text': [], 'label': []}
train_augmented_bert = {'text': [], 'label': []}
train_augmented_hybrid = {'text': [], 'label': []}

train_datasets = [
    train_augmented_wordnet,
    train_augmented_lesk,
    train_augmented_bert,
    train_augmented_hybrid,
    train_original
]

In [ ]:
for row in tqdm(original_dataset['train']):
    s1 = row['sentence1']
    s2 = row['sentence2']
    label = row['label']

    prompt = convert_to_prompt(s1, s2)
    for ds in train_datasets:
        ds['text'].append(prompt)
        ds['label'].append(label)
    
    for ds, aug_fn in zip(train_datasets, [augment_wordnet, augment_lesk, augment_bert, augment_hybrid]):
        aug_prompt = convert_to_prompt(
            aug_fn(s1),
            aug_fn(s2)
        )
        if aug_prompt != prompt:
            ds['text'].append(aug_prompt)
            ds['label'].append(label)

In [ ]:
val = {'text': [], 'label': []}
test = {'text': [], 'label': []}

for ds, split in zip([val, test], ['validation', 'test']):
    for row in original_dataset[split]:
        s1 = row['sentence1']
        s2 = row['sentence2']
        label = row['label']
        
        prompt = convert_to_prompt(s1, s2)
        ds['text'].append(prompt)
        ds['label'].append(label)

In [ ]:
DatasetDict({
    'train-original': Dataset.from_dict(train_original),
    'train-augmented-wordnet': Dataset.from_dict(train_augmented_wordnet),
    'train-augmented-lesk': Dataset.from_dict(train_augmented_lesk),
    'train-augmented-bert': Dataset.from_dict(train_augmented_bert),
    'train-augmented-hybrid': Dataset.from_dict(train_augmented_hybrid),
    'validation': Dataset.from_dict(val),
    'test': Dataset.from_dict(test)
}).save_to_disk('dataset.hf')